<br>

<div style="text-align: center">Confidential – Qualcomm Technologies, Inc. and/or its affiliated companies – May Contain Trade Secrets</div>

<div style="text-align: center; font-weight: bold">MAY CONTAIN U.S. AND INTERNATIONAL EXPORT CONTROLLED INFORMATION</div> <br>

**Takeaways:** Users will learn how to compile and execute the BERT base neural network model on Qualcomm® Cloud AI (AIC) Inference Accelerator

**Before you start:** There are some commands (folder locations etc) that will need to be updated in this notebook based on your platform and installation location. Some commands might need sudo prefix to run properly.

**Last Verified Qualcomm Cloud AI Platform SDK and Apps SDK Version:** 1.9.1.25 

## Qualcomm Cloud AI Getting started with BERT base

BERT (Bidirectional Encoder Representations from Transformers) is a neural network-based model that uses a transformer architecture. It is designed to understand the context and meaning of words in a sentence by capturing bidirectional dependencies.

BERT has several variants, including BERT Base and BERT Large, which differ in the size and number of parameters.
* BERT Base: Number of Layers L=12, Size of the hidden layer, H=768, and Self-attention heads, A=12 with Total Parameters=110M
* BERT Large: Number of Layers L=24, Size of the hidden layer, H=1024, and Self-attention heads, A=16 with Total Parameters=340M

In this example we will download a BERT base model, compile the model in FP16 precision, then run the model on the Cloud AI (AIC) accelerator card.

The following topics are covered in this notebook:

### BERT base Example
   > 1) Setup and Download the model
   > 2) Model Preparation
   > 3) Input data generation
   > 4) Compilation
   > 5) Inferencing (inference/sec)
   > 6) End-to-End Latency

### Setup and Download the Model
For setup, install the required Python packages mentioned under requirements.txt and then download the pretrained bert-base model.

In [1]:
!cat bertbase_setup.sh

pip install -r requirements.txt

if [ -d generatedModels ]
then
  rm -rf generatedModels
fi

optimum-cli export onnx --model bert-base-cased --cache_dir model_files/cased --opset 11 --task question-answering generatedModels/ONNX/cased


In [2]:
!bash bertbase_setup.sh

2023-05-31 19:19:33.465927: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: :/opt/qti-aic/dev/lib/x86_64
2023-05-31 19:19:33.465969: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2023-05-31 19:19:37.613874: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: :/opt/qti-aic/dev/lib/x86_64
2023-05-31 19:19:37.613917: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
Framework not specified. Using pt to export to ONNX.
Some weights of the model checkpoint at bert-base-cased were not used when in

Here's the BERT base ONNX model.

In [3]:
!ls ./generatedModels/ONNX/cased

config.json  special_tokens_map.json  tokenizer_config.json
model.onnx   tokenizer.json	      vocab.txt


Here's the BERT configuration we'll be working with.

In [4]:
!cat ./generatedModels/ONNX/cased/config.json

{
  "_name_or_path": "bert-base-cased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.29.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 28996
}


### Model Preparation

* Modify the onnx file to handle constants > FP16_Max and < FP16_Min
* Run model preparator tool: The objective of this tool is to make sure model functionality and it performs well on the Cloud AI (AIC) accelerator card.

#### Modify the onnx file to handle constants

Find constants which are out of FP16 range and clip to FP16 range.

In [5]:
import onnx
from onnx import numpy_helper
import numpy as np

model_card = 'bert-base-cased'
gen_models_path = f"generatedModels/ONNX/cased"
model_base_name = model_card

def fix_onnx_fp16(
    gen_models_path: str,
    model_base_name: str,
) -> str:
    finfo = np.finfo(np.float16)
    fp16_max = finfo.max
    fp16_min = finfo.min

    model = onnx.load(f"generatedModels/ONNX/cased/model.onnx")
    fp16_fix = False
    for tensor in onnx.external_data_helper._get_all_tensors(model):
        nptensor = numpy_helper.to_array(tensor, gen_models_path)
        if nptensor.dtype == np.float32 and (
            np.any(nptensor > fp16_max) or np.any(nptensor < fp16_min)
        ):
            # print(f'tensor value : {nptensor} above {fp16_max} or below {fp16_min}')
            nptensor = np.clip(nptensor, fp16_min, fp16_max)
            new_tensor = numpy_helper.from_array(nptensor, tensor.name)
            tensor.CopyFrom(new_tensor)
            fp16_fix = True
            
    if fp16_fix:
        # Save FP16 model
        print("Found constants out of FP16 range, clipped to FP16 range")
        model_base_name += "_fix_outofrange_fp16"
        onnx.save(model, f=f"{gen_models_path}/{model_base_name}.onnx")
        print(f"Saving modified onnx file at {gen_models_path}/{model_base_name}.onnx")
    return model_base_name

fp16_model_name = fix_onnx_fp16(gen_models_path=gen_models_path, model_base_name=model_base_name)

Found constants out of FP16 range, clipped to FP16 range
Saving modified onnx file at generatedModels/ONNX/cased/bert-base-cased_fix_outofrange_fp16.onnx


In [6]:
!ls ./generatedModels/ONNX/cased

bert-base-cased_fix_outofrange_fp16.onnx  special_tokens_map.json  vocab.txt
config.json				  tokenizer.json
model.onnx				  tokenizer_config.json


#### Run model preparator tool

The objective of this tool is to make sure model functionality and it performs well on the Cloud AI (AIC) accelerator card. Generally, trained models may have control flow operators, dynamic tensors, and other inference unfriendly operators or subgraphs. This tool applies precompile optimization and cleans the model.

The parameters required by the tool are configured in a configuration file in YAML format (**bertbase.yaml**) and the tool parses this configuration file to execute.

In [7]:
!cat bertbase.yaml

##############################################################################
# Model Preparation configuration file.
# Official Model Location: https://huggingface.co/models
##############################################################################
MODEL:
    INFO:
        DESCRIPTION: "HuggingFace BERT base cased Model"
        MODEL_TYPE: TRANSFORMERS
        MODEL_PATH: 'generatedModels/ONNX/cased/bert-base-cased_fix_outofrange_fp16.onnx'
        INPUT_INFO: [["input_ids", [1, 128]],
                     ["attention_mask", [1, 128]],
                     ["token_type_ids", [1, 128]]]
        EXPORT_TYPE: ONNX
        DYNAMIC_INFO: []
        VALIDATE: False
        WORKSPACE: WORKSPACE
        VERBOSE: INFO #INFO, DEBUG, WARNING. TRACE

    PRE_POST_HANDLE:
        ANCHOR_BIN_FILE: None
        POST_PLUGIN: NONE
        PRE_PLUGIN: False
        NMS_PARAMS:
            MAX_OUTPUT_SIZE_PER_CLASS: None
            MAX_TOTAL_SIZE: None
            IOU_THRESHOLD: None
            

In [8]:
!cat model-preparator.sh

###############################################################################
# Run model preparator tool.
###############################################################################

#!/bin/bash
#complete modelpath
model_output_path=./generatedModels/ONNX/cased/prepmodel.onnx

#preparing the model as per AIC
prepare_model(){
	if [ -d WORKSPACE ]
	then
		rm -rf WORKSPACE
	fi

	python3 /opt/qti-aic/tools/qaic-pytools/qaic-model-preparator.py --config bertbase.yaml
	python3 generateModel.py --model-ip-path  WORKSPACE/bert-base-cased_fix_outofrange_fp16_preparator_aic100.onnx --model-op-path ${model_output_path}

}

prepare_model


In [9]:
!bash model-preparator.sh

2023-05-31 19:21:25.271 | WARNING  | __main__:prepare_work_dir:35 - Directory WORKSPACE will be deleted if already exists. Take backup before execution.
2023-05-31 19:21:25.276 | INFO     | __main__:main:78 - QAic Model Preparator Tool
2023-05-31 19:21:25.627 | INFO     | qaic_pytools.qmodel.backend.onnx.model:native_checker:638 - Graph checker passed successfully.
2023-05-31 19:21:25.937 | INFO     | qaic_pytools.qmodel.preparator.preparator:__prepare_onnx:215 - Native Checker on the input model is successful
2023-05-31 19:21:25.938 | INFO     | qaic_pytools.qmodel.preparator.preparator:__prepare_onnx:219 - Native Checker on the model is successful
2023-05-31 19:21:26.939 | INFO     | qaic_pytools.qmodel.preparator.preparator:__prepare_onnx:231 - Internal Checker (Ort) on the model is successful
2023-05-31 19:21:27.147 | WARNING  | qaic_pytools.qmodel.backend.onnx.onnx_model:native_shape_inference:613 - Symbolic shape inference failed. Exception: Incomplete symbolic shape inference. R

In [10]:
!ls ./generatedModels/ONNX/cased

bert-base-cased_fix_outofrange_fp16.onnx  special_tokens_map.json
config.json				  tokenizer.json
model.onnx				  tokenizer_config.json
prepmodel.onnx				  vocab.txt


### Input data generation
In this step we generate random inputs for BERT base model using **input-data-generation.sh** script, which will be used during inference request.

In [11]:
!cat input-data-generation.sh

###############################################################################
# Generate random inputs for BERT base model
###############################################################################

#!/bin/bash
#Input data generation
python -W ignore generateInputs.py --seq_len 128 --batch_size=1


In [12]:
!bash input-data-generation.sh

Inputs are generated under **inputFiles** folder.

In [13]:
!ls ./inputFiles

input_ids_sl128_bs1.raw  input_mask_sl128_bs1.raw  segment_ids_sl128_bs1.raw


### Compile the Network
In this step we generate Cloud AI (AIC) network binaries from the ONNX model using the `qaic-exec` compiler.  The binaries are generated in the `./compiler_output_FP16` folder. The network is compiled with FP16 precision and following parameters.
   > 1) Cores - Network will execute on 1 NSP compute cores
   > 2) Batchsize - batchsize of 1
   > 3) All other performnace tuning parameters are set to default. 

In [ ]:
!rm -rf ./compiler_output_FP16

!/opt/qti-aic/exec/qaic-exec \
    -model=./generatedModels/ONNX/cased/prepmodel.onnx \
    -aic-hw-version=2.0 \
    -aic-hw \
    -convert-to-fp16 \
    -aic-num-cores=1 \
    -onnx-define-symbol=batch_size,1  \
    -stats-batchsize=1 \
    -onnx-define-symbol=seq_len,128 \
    -multicast-weights \
    -aic-binary-dir=./compiler_output_FP16 \
    -compile-only

Reading ONNX Model from ./generatedModels/ONNX/cased/prepmodel.onnx
loading compiler from: /opt/qti-aic/dev/lib/x86_64/libQAicCompiler.so
Compile started ............... 
Compiling model with FP16 precision.


### Inferencing
Next we run the compiled network binaries on Cloud AI (AIC) hardware with the `qaic-runner` tool to check the throughput.  We use one of the input sequences generated during the 'Generate Input Data' step above.

Inferencing runs on `device id 0 by default`.  To specify a different card, add the `--aic-device-id` option.  For example: `--aic-device-id 1` to run on device 1.  Run the `/opt/qti-aic-tools/qaic-util -q` tool to list the device IDs.

In [20]:
!/opt/qti-aic/exec/qaic-runner \
    --test-data ./compiler_output_FP16 \
    --aic-num-of-activations 1 \
    --set-size 10 \
    --num-iter 1000 \
    --input-file inputFiles/input_ids_sl128_bs1.raw \
    --input-file inputFiles/input_mask_sl128_bs1.raw \
    --input-file inputFiles/segment_ids_sl128_bs1.raw \
    --aic-device-id 0

loading /opt/qti-aic/dev/lib/x86_64/libQAic.so
Input file: inputFiles/input_ids_sl128_bs1.raw
Input file: inputFiles/input_mask_sl128_bs1.raw
Input file: inputFiles/segment_ids_sl128_bs1.raw
 ---- Stats ----
InferenceCnt 1000 TotalDuration 7824571us BatchSize 1 Inf/Sec 127.803


### End-to-End Latency

Latency statistics are available from the `qaic-runner` tool with the following options:

```
Usage: qaic-runner [options]
  -S, --set-size <i>                 Set Size for inference loop execution, default:10, min:1
  -l, --live-reporting               Enable Live reporting periodic at 1 sec interval, default off
  -s, --stats                        Enable Live Profiling Stats reporting periodically at 1 sec interval
```

End-to-end latency is calculated as:<br>
**preProcTime + execTotalTime + postProcTime**

Set size of 1 **--set-size 1** is recommended for latency measurement.  

<img src="./images/latency_e2e.png" alt="Latency Breakdown" width = 800 />

In [21]:
# Collect latency stats output for <max_run> runs

max_run = 3
results = [None] * max_run

for idx in range(0, max_run):
    print('Run {}'.format(idx))
    results[idx] = !/opt/qti-aic/exec/qaic-runner \
        --test-data ./compiler_output_FP16 \
        --aic-num-of-activations 1 \
        --num-iter 1000 \
        --input-file inputFiles/input_ids_sl128_bs1.raw \
        --input-file inputFiles/input_mask_sl128_bs1.raw \
        --input-file inputFiles/segment_ids_sl128_bs1.raw \
        --aic-device-id 0 \
        --set-size 1 \
        --live-reporting 1000 \
        --stats

Run 0
Run 1
Run 2


Define a helper class QAicLatencyStats to parse the latency stats.

In [22]:
# Define a helper class QAicLatencyStats to parse the latency stats.
import re
import pandas as pd

class QAicLatencyStats:
    def __init__(self):
        pass
    
    def load(self, results):
        measurement_map = {'preProcessingLatencyStats': 'pre_proc_us',
                           'execTotalTimeStats': 'exec_total_us',
                           'execKernelToCompletStats': 'exec_kernel_to_complete_us',
                           'postProcessingLatencyStats': 'post_proc_us'}

        columns = ['activation', 'num_samples', 'timestamp_ms', 'num_inf_completed', 'rate_inf_sec']
        data = {}
        for column in columns:
            data[column] = []

        # Add measurement columns
        for key,value in measurement_map.items():
            data[value] = []
        
        for result in results:
            match = re.match('Activation\[([0-9]+)\]', result)
            if match:
                activation = int(match.group(1))
                #print('activation {}'.format(activation))
                data['activation'].append(activation)

            match = re.match('NumSamples\:\s+([0-9]+)', result)
            if match:
                num_samples = int(match.group(1))
                #print('num_samples {}'.format(num_samples))
                data['num_samples'].append(num_samples)

            match = re.match('([\w]+) \(us\).+avg\:\s+([0-9\.]+)', result)
            if match:
                measurement = match.group(1)
                if measurement in measurement_map.keys():
                    measurement = measurement_map[measurement]
                    latency = float(match.group(2))
                    #print('{} latency {}'.format(measurement, latency))
                    data[measurement].append(latency)

            #timestamp:    9000ms numInfCompleted:      881 rate:   97.889 Inf/Sec
            match = re.match('timestamp\:\s+([0-9]+)ms\s+numInfCompleted\:\s+([0-9]+)\s+rate\:\s+([0-9\.]+) Inf\/Sec', result)
            if match:
                timestamp = int(match.group(1))
                num_inf_completed = int(match.group(2))
                rate_inf_sec = float(match.group(3))
                #print('timestamp {}'.format(timestamp))
                data['timestamp_ms'].append(timestamp)
                data['num_inf_completed'].append(num_inf_completed)
                data['rate_inf_sec'].append(rate_inf_sec)
                                
        df = pd.DataFrame.from_dict(data)
        df['e2e_total_latency_us'] = df['pre_proc_us'] + df['exec_total_us'] + df['post_proc_us']
        return df

Parse and show the average latency results.

In [23]:
#import QAicLatencyStats from get-latency-stat

latency_stats = QAicLatencyStats()

avg = 0
df = pd.DataFrame()
for idx in range(0, max_run):
    run_df = latency_stats.load(results[idx])
    avg += run_df['e2e_total_latency_us'].mean()
    run_df['run'] = idx
    df = pd.concat([run_df, df])

print('Average e2e latency: {:0.2f} us'.format(avg / max_run))

Average e2e latency: 7903.53 us


<br>
<div style="text-align: center">Confidential – Qualcomm Technologies, Inc. and/or its affiliated companies – May Contain Trade Secrets</div>

<div style="text-align: center; font-weight: bold">MAY CONTAIN U.S. AND INTERNATIONAL EXPORT CONTROLLED INFORMATION</div>